In [ ]:
import tiktoken
import re

In [ ]:
with open("the-verdict.txt", "r") as f:
  raw_text = f.read()

In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

all_words = sorted(set(preprocessed))

vocab = {token:integer for integer,token in enumerate(all_words)}

# special tokens
special_tokens = ["<|endoftext|>", "<|unk|>"]

for i, t in enumerate(special_tokens):
  vocab[t] = len(vocab)

print(len(vocab))

1132


In [ ]:
class SimpleTokenizer:
  def __init__(self, vocab):
    self.str_to_int = vocab
    self.int_to_str = {i: s for s, i in vocab.items()}

  def encode(self, text):
    preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    preprocessed = [item.strip() for item in preprocessed if item.strip()]
    preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
    ids = [self.str_to_int.get(s, self.str_to_int.get("<|unk|>")) for s in preprocessed]

    return ids

  def decode(self, ids):
    text = " ".join([self.int_to_str[i] for i in ids])
    text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)

    return text

In [ ]:
text = "Hello, do you like coffee?"

tokenizer = SimpleTokenizer(vocab)

ids = tokenizer.encode(text)
print(ids)

decoded = tokenizer.decode(ids)
print(decoded)

[1131, 5, 355, 1126, 628, 1131, 10]
<|unk|>, do you like <|unk|>?


In [ ]:
# tiktoken with unknown text

text = "Akwirw ier"
tokenizer = tiktoken.get_encoding("gpt2")

ids = tokenizer.encode(text)
print(ids)

decoded = tokenizer.decode(ids)
print(decoded)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


In [ ]:
enc_raw_text = tokenizer.encode(raw_text)

enc_sample = enc_raw_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[:context_size + 1]

for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
print(context, "---->", desired)

for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

[290, 4920, 2241, 287] ----> 257
 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt)
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i + max_length]
      target_chunk = token_ids[i + 1: i + max_length + 1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDataset(txt, tokenizer, max_length, stride)

  dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

  return dataloader

In [ ]:
dataloader = create_dataloader(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [ ]:
# example encoding to embedding

vocab_size = 50257 # dependent on the tokenizer with encoding gpt2
output_dim = 256

torch.manual_seed(123)
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim) # gpt3 is 12,288

max_length = 4
dataloader = create_dataloader(
raw_text, batch_size=8, max_length=max_length,
stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [ ]:
# for abs positional embeddings like in gpt3, have an additional layer

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)
print(pos_embeddings)

torch.Size([4, 256])
tensor([[-1.4150, -0.3142,  0.2827,  ...,  0.8155, -0.1085, -1.1927],
        [-1.9800,  0.0610, -0.0494,  ..., -0.6422,  0.5716, -1.1329],
        [ 1.0052,  1.7802,  1.2652,  ..., -1.1619, -0.1109,  1.0411],
        [ 0.3760, -0.3758, -0.0484,  ...,  0.1080,  0.3852,  1.0876]],
       grad_fn=<EmbeddingBackward0>)


In [ ]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)
print(input_embeddings)

torch.Size([8, 4, 256])
tensor([[[-1.4789e+00,  1.7579e-02,  3.8972e-01,  ...,  1.3504e+00,
          -9.1092e-01, -3.5165e+00],
         [-2.3324e+00,  4.1186e-01,  9.3785e-01,  ..., -2.4888e+00,
          -1.1318e+00, -8.1060e-01],
         [ 2.0069e+00,  2.7101e+00,  1.8634e-03,  ..., -2.3875e+00,
           1.0070e+00,  1.1753e+00],
         [ 1.1756e+00,  1.9079e+00, -7.0086e-01,  ..., -1.0136e+00,
           8.5575e-01,  1.2408e+00]],

        [[-1.5232e+00, -1.5865e+00, -9.3895e-01,  ..., -1.0444e-01,
           1.8989e+00, -2.6065e+00],
         [-2.2227e+00,  9.7547e-01,  1.0391e+00,  ..., -1.5073e+00,
           4.0985e+00, -4.1040e-01],
         [ 4.6181e-01,  3.4006e+00,  2.4874e+00,  ..., -4.8047e-01,
          -1.5141e+00,  1.1903e+00],
         [ 2.5676e-02, -1.3083e+00, -1.3384e+00,  ..., -1.3900e+00,
           5.2514e-01,  1.4606e+00]],

        [[-1.3473e+00, -1.3899e+00,  4.1981e-01,  ...,  4.2434e-01,
          -1.2059e+00, -4.3363e-01],
         [-3.2361e+00,  3.7

In [ ]:
# attention
inputs = torch.tensor(
[[0.43, 0.15, 0.89],
 [0.55, 0.87, 0.66],
 [0.57, 0.85, 0.64],
 [0.22, 0.58, 0.33],
 [0.77, 0.25, 0.10],
 [0.05, 0.80, 0.55]]
)

query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

# normalize the scores to get weights (here softmax may be used)
attn_weights = torch.softmax(attn_scores_2, dim=-1) # instructing the softmax function to apply the normalization along the last dimension
print(attn_weights)
print(sum(attn_weights))

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [ ]:
# self-attention (trainable weight matrices so that the attention layer produces good context vectors)

x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(123)
# set requires_grad to True when training
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

# weight parameters are the fundamental, learned coefficients that define
# the network’s connections, while attention weights are dynamic, context-specific values.

tensor([0.4306, 1.4551])


In [ ]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [ ]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [ ]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [ ]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


In [ ]:
import torch.nn as nn

class SelfAttention(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    keys = x @ self.W_key
    queries = x @ self.W_query
    values = x @ self.W_value
    attn_scores = queries @ keys.T # omega
    attn_weights = torch.softmax(
    attn_scores / keys.shape[-1]**0.5, dim=-1
    )
    context_vec = attn_weights @ values
    return context_vec

In [ ]:
context_length = attn_scores_2.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1385, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1385, 0.2379, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1385, 0.2379, 0.2333, 0.0000, 0.0000, 0.0000],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.0000, 0.0000],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.0000],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]])


In [ ]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3680, 0.6320, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2272, 0.3902, 0.3826, 0.0000, 0.0000, 0.0000],
        [0.1888, 0.3242, 0.3179, 0.1690, 0.0000, 0.0000],
        [0.1646, 0.2826, 0.2771, 0.1473, 0.1285, 0.0000],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]])
